In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (df_metric, df_trace, df_log, df_error):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Incident window (UTC, inclusive)
window_start = pd.Timestamp("2024-01-24 10:00:25", tz="UTC")
window_end = pd.Timestamp("2024-01-24 10:30:25", tz="UTC")

# --- metric.csv summary ---
# Aggregate full-series stats per (cmdb_id, kpi_name)
metric_grp = df_metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)
metric_agg_full = metric_grp.agg(
    total_count=('value', 'size'),
    p95_value=('value', lambda s: float(s.quantile(0.95))),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max')
).reset_index()

# Count rows within the incident window (must be computed after global stats)
mask_metric_window = (df_metric['timestamp'] >= window_start) & (df_metric['timestamp'] <= window_end)
metric_window_counts = df_metric[mask_metric_window].groupby(['cmdb_id', 'kpi_name']).size().rename('count_in_window').reset_index()
metric_summary = metric_agg_full.merge(metric_window_counts, on=['cmdb_id', 'kpi_name'], how='left')
metric_summary['count_in_window'] = metric_summary['count_in_window'].fillna(0).astype(int)

# Prepare top 20 by count_in_window
metric_summary_top20 = metric_summary.sort_values('count_in_window', ascending=False).head(20).copy()

# Unique cmdb_ids in metric
metric_cmdbs = pd.Series(df_metric['cmdb_id'].dropna().unique(), name='cmdb_id')

# --- trace.csv summary ---
trace_grp = df_trace.groupby(['cmdb_id', 'trace_name'], dropna=False)
trace_agg_full = trace_grp.agg(
    total_count=('value', 'size'),
    p95_value=('value', lambda s: float(s.quantile(0.95))),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max')
).reset_index()

mask_trace_window = (df_trace['timestamp'] >= window_start) & (df_trace['timestamp'] <= window_end)
trace_window_counts = df_trace[mask_trace_window].groupby(['cmdb_id', 'trace_name']).size().rename('count_in_window').reset_index()
trace_summary = trace_agg_full.merge(trace_window_counts, on=['cmdb_id', 'trace_name'], how='left')
trace_summary['count_in_window'] = trace_summary['count_in_window'].fillna(0).astype(int)

trace_summary_top20 = trace_summary.sort_values('count_in_window', ascending=False).head(20).copy()
trace_cmdbs = pd.Series(df_trace['cmdb_id'].dropna().unique(), name='cmdb_id')

# --- log.csv summary ---
log_grp = df_log.groupby(['cmdb_id', 'log_name'], dropna=False)
log_agg_full = log_grp.agg(
    total_count=('value', 'size'),
    p95_value=('value', lambda s: float(s.quantile(0.95))),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max')
).reset_index()

mask_log_window = (df_log['timestamp'] >= window_start) & (df_log['timestamp'] <= window_end)
log_window_counts = df_log[mask_log_window].groupby(['cmdb_id', 'log_name']).size().rename('count_in_window').reset_index()
log_summary = log_agg_full.merge(log_window_counts, on=['cmdb_id', 'log_name'], how='left')
log_summary['count_in_window'] = log_summary['count_in_window'].fillna(0).astype(int)

log_summary_top20 = log_summary.sort_values('count_in_window', ascending=False).head(20).copy()
log_cmdbs = pd.Series(df_log['cmdb_id'].dropna().unique(), name='cmdb_id')

# --- error_logs.csv summary ---
total_error_count = len(df_error)
mask_error_window = (df_error['timestamp'] >= window_start) & (df_error['timestamp'] <= window_end)
error_in_window_df = df_error[mask_error_window].copy().sort_values('timestamp').reset_index(drop=True)
error_in_window_count = len(error_in_window_df)

# Up to 20 sample rows within the window (timestamp, cmdb_id, message)
error_samples = error_in_window_df[['timestamp', 'cmdb_id', 'message']].head(20).copy()

# Compact summary for error counts
error_summary_info = pd.DataFrame({
    'total_error_count': [total_error_count],
    'error_count_in_window': [error_in_window_count]
})

# Display results (variables returned)
metric_summary_top20, metric_cmdbs, trace_summary_top20, trace_cmdbs, log_summary_top20, log_cmdbs, error_summary_info, error_samples

```
Out[1]:
```
summary = (
    "Summary of telemetry for incident window 2024-01-24 10:00:25 to 2024-01-24 10:30:25 UTC:\n"
    "- metric.csv: 68 unique cmdb_id values. Many metric groups have full coverage in the window (count_in_window = 25). "
    "Top groups in the returned list include ts-admin-basic-info-service (cpu, mem, diskio, socket, latency), ts-travel2-service and ts-travel-service (cpu, mem, diskio, latency, socket), and ts-voucher-service (socket). Global p95 values were computed over each group's entire series before window filtering.\n"
    "- trace.csv: 28 unique cmdb_id values. Trace groups also show full coverage in the window (count_in_window = 25 for top entries). Notable traces include calls to/from user-service and auth-service and traces to admin-travel-service (example: duration_p95 for admin-travel was ~4.54s in the aggregated results).\n"
    "- log.csv: 45 unique cmdb_id values. log.row_count and log.error_count groups are present with full coverage in the window. Several services show high log.row_count p95 (examples: ts-basic-service, ts-order-service).\n"
    "- error_logs.csv: no error log rows present (total_error_count = 0, error_count_in_window = 0).\n\n"
    "Overall: telemetry is available and consistent across metric/trace/log sources for the window. There are no captured error log entries. The most notable signals to investigate further (based on p95 values and metric types) are ts-admin-basic-info-service, ts-travel2-service / ts-travel-service, and ts-voucher-service — particularly their latency/duration, memory, and socket-related metrics. "
    "These are pointers for deeper investigation rather than confirmed root causes."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  total_count     p95_value             min_timestamp             max_timestamp  count_in_window
360           ts-voucher-service      socket           25  4.000000e+00 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
0    ts-admin-basic-info-service         cpu           25  8.888737e-01 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
1    ts-admin-basic-info-service      diskio           25  5.787439e+04 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
2    ts-admin-basic-info-service       error           25  0.000000e+00 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
3    ts-admin-basic-info-service  latency-50           25  1.733192e-02 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
4    ts-admin-basic-info-service  latency-90           25  2.357821e-02 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
5    ts-admin-basic-info-service         mem           25  2.313912e+08 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
6    ts-admin-basic-info-service      socket           25  8.010000e+00 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
7    ts-admin-basic-info-service    workload           25  6.928933e-01 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
321             ts-travel2-mongo         mem           25  6.710697e+07 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
322             ts-travel2-mongo      socket           25  8.000000e+00 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
323           ts-travel2-service         cpu           25  5.274661e+00 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
324           ts-travel2-service      diskio           25  5.988948e+04 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
325           ts-travel2-service  latency-50           25  3.232860e-01 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
326           ts-travel2-service  latency-90           25  3.688510e+00 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
327           ts-travel2-service         mem           25  2.759753e+08 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
328           ts-travel2-service      socket           25  3.176513e+01 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
313            ts-travel-service       error           25  0.000000e+00 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
314            ts-travel-service  latency-50           25  3.144030e-01 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
315            ts-travel-service  latency-90           25  2.196402e+00 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25, 0      ts-admin-basic-info-service
1           ts-admin-order-service
2           ts-admin-route-service
3          ts-admin-travel-service
4            ts-admin-user-service
                  ...             
63                   ts-user-mongo
64                 ts-user-service
65    ts-verification-code-service
66                ts-voucher-mysql
67              ts-voucher-service
Name: cmdb_id, Length: 68, dtype: object,                 cmdb_id                                         trace_name  total_count   p95_value             min_timestamp             max_timestamp  count_in_window
791     ts-user-service                 trace.to_ts-user-service.row_count           25   21.600000 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
0                  root  trace.to_ts-admin-basic-info-service.duration_...           25    0.011059 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
1                  root  trace.to_ts-admin-basic-info-service.duration_p95           25    0.016275 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
2                  root    trace.to_ts-admin-basic-info-service.error_rate           25    0.000000 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
3                  root     trace.to_ts-admin-basic-info-service.row_count           25   21.800000 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
4                  root     trace.to_ts-admin-travel-service.duration_mean           25    4.264594 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
5                  root      trace.to_ts-admin-travel-service.duration_p95           25    4.541190 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
6                  root        trace.to_ts-admin-travel-service.error_rate           25    0.000000 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
7                  root         trace.to_ts-admin-travel-service.row_count           25   10.800000 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
790     ts-user-service                trace.to_ts-user-service.error_rate           25    0.000000 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
789     ts-user-service              trace.to_ts-user-service.duration_p95           25    0.003624 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
788     ts-user-service             trace.to_ts-user-service.duration_mean           25    0.002333 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
787     ts-user-service               trace.from_ts-user-service.row_count           25   21.600000 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
12                 root             trace.to_ts-auth-service.duration_mean           25    0.346447 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
13                 root              trace.to_ts-auth-service.duration_p95           25    0.883813 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
14                 root                trace.to_ts-auth-service.error_rate           25    0.000000 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
15                 root                 trace.to_ts-auth-service.row_count           25   86.800000 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
786     ts-user-service              trace.from_ts-user-service.error_rate           25    0.000000 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
746  ts-travel2-service              trace.from_ts-seat-service.error_rate           25    0.000000 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
747  ts-travel2-service               trace.from_ts-seat-service.row_count           25  366.200000 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25, 0                            root
1     ts-admin-basic-info-service
2         ts-admin-travel-service
3            ts-assurance-service
4                 ts-auth-service
                 ...             
23          ts-ticketinfo-service
24               ts-train-service
25              ts-travel-service
26             ts-travel2-service
27                ts-user-service
Name: cmdb_id, Length: 28, dtype: object,                         cmdb_id         log_name  total_count  p95_value             min_timestamp             max_timestamp  count_in_window
0   ts-admin-basic-info-service  log.error_count           25        0.0 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
1   ts-admin-basic-info-service    log.row_count           25       32.6 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
2       ts-admin-travel-service  log.error_count           25        0.0 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
3       ts-admin-travel-service    log.row_count           25       32.4 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
24          ts-contacts-service  log.error_count           25        0.0 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
25          ts-contacts-service    log.row_count           25       64.8 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
13             ts-basic-service    log.row_count           25     2811.2 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
12             ts-basic-service  log.error_count           25        0.0 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
11              ts-auth-service    log.row_count           25      347.2 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
10              ts-auth-service  log.error_count           25        0.0 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
61             ts-price-service    log.row_count           25      312.4 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
60             ts-price-service  log.error_count           25        0.0 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
64             ts-route-service  log.error_count           25        0.0 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
65             ts-route-service    log.row_count           25     1093.6 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
70          ts-security-service  log.error_count           25        0.0 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
71          ts-security-service    log.row_count           25       86.4 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
47             ts-order-service    log.row_count           25      415.2 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
46             ts-order-service  log.error_count           25        0.0 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
45       ts-order-other-service    log.row_count           25      517.2 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25
44       ts-order-other-service  log.error_count           25        0.0 2024-01-24 10:03:00+00:00 2024-01-24 10:27:00+00:00               25, 0     ts-admin-basic-info-service
1         ts-admin-travel-service
2            ts-assurance-service
3                   ts-auth-mongo
4                 ts-auth-service
                 ...             
40                 ts-price-mongo
41              ts-security-mongo
42         ts-ticket-office-mongo
43                ts-travel-mongo
44               ts-travel2-mongo
Name: cmdb_id, Length: 45, dtype: object,    total_error_count  error_count_in_window
0                  0                      0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure timestamps parsed (idempotent)
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Target components to analyze
components = [
    "ts-auth-service",
    "ts-order-service",
    "ts-route-service",
    "ts-train-service",
    "ts-travel-service"
]

# Incident window (UTC, inclusive)
window_start = pd.Timestamp("2024-01-24 10:00:25", tz="UTC")
window_end = pd.Timestamp("2024-01-24 10:30:25", tz="UTC")

# Restrict metric series to the specified components
df_m = df_metric[df_metric['cmdb_id'].isin(components)].copy()

# If no data for these components, return empty results gracefully
if df_m.shape[0] == 0:
    groups_with_anomalies = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_count','count_in_window','p95_value',
        'number_of_anomalies','earliest_anomaly_timestamp','max_value_in_window'
    ])
    earliest_anomalies = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value'])
else:
    # 1) Compute global p95 and total_count per (cmdb_id, kpi_name) using full available series
    grp = df_m.groupby(['cmdb_id', 'kpi_name'], dropna=False)
    grp_stats = grp['value'].agg(
        total_count='size',
        p95_value=lambda s: float(s.quantile(0.95))
    ).reset_index()

    # 2) Filter rows to the incident window
    mask_window = (df_m['timestamp'] >= window_start) & (df_m['timestamp'] <= window_end)
    df_window = df_m[mask_window].copy()

    # 3) Compute count_in_window per group
    count_in_window = df_window.groupby(['cmdb_id','kpi_name']).size().rename('count_in_window').reset_index()

    # Merge group stats with window counts
    groups = grp_stats.merge(count_in_window, on=['cmdb_id','kpi_name'], how='left')
    groups['count_in_window'] = groups['count_in_window'].fillna(0).astype(int)

    # Determine anomalies: rows in window where value >= group's p95_value
    # Merge p95 into window rows for comparison
    df_window_with_p95 = df_window.merge(grp_stats[['cmdb_id','kpi_name','p95_value']], on=['cmdb_id','kpi_name'], how='left')

    # Mark anomalies
    df_window_with_p95['is_anomaly'] = df_window_with_p95['value'] >= df_window_with_p95['p95_value']

    # number_of_anomalies per group
    anomalies_count = df_window_with_p95[df_window_with_p95['is_anomaly']].groupby(
        ['cmdb_id','kpi_name']
    ).agg(
        number_of_anomalies=('is_anomaly','sum'),
        earliest_anomaly_timestamp=('timestamp','min'),
        max_value_in_window=('value','max')
    ).reset_index()

    # Merge anomaly summaries back into groups
    groups = groups.merge(anomalies_count, on=['cmdb_id','kpi_name'], how='left')
    groups['number_of_anomalies'] = groups['number_of_anomalies'].fillna(0).astype(int)
    groups['earliest_anomaly_timestamp'] = groups['earliest_anomaly_timestamp']  # may contain NaT
    groups['max_value_in_window'] = groups['max_value_in_window'].fillna(pd.NA)

    # Keep only groups with number_of_anomalies > 0
    groups_with_anomalies = groups[groups['number_of_anomalies'] > 0].copy()

    # Sort by number_of_anomalies desc then max_value_in_window desc
    groups_with_anomalies = groups_with_anomalies.sort_values(
        by=['number_of_anomalies','max_value_in_window'], ascending=[False, False]
    ).head(20)

    # Earliest 10 anomaly rows across all these groups (timestamp, cmdb_id, kpi_name, value)
    anomalies_all = df_window_with_p95[df_window_with_p95['is_anomaly']].copy()
    earliest_anomalies = anomalies_all[['timestamp','cmdb_id','kpi_name','value']].sort_values(
        'timestamp', ascending=True
    ).head(10).reset_index(drop=True)

# Display compact results
groups_with_anomalies, earliest_anomalies

```
Out[2]:
```
summary = (
    "Analysis restricted to the five candidate services (ts-auth-service, ts-order-service, "
    "ts-route-service, ts-train-service, ts-travel-service) for the incident window 2024-01-24 10:00:25–10:30:25 UTC:\n\n"
    "- ts-travel-service: highest signal count. It shows 25 hits for the 'error' metric (p95=0 so every 0 value qualifies) and multiple anomalies in mem, diskio, socket, and workload. Earliest anomalies appear at 10:03:00 UTC.\n"
    "- Memory spikes (mem) were observed in several services: ts-travel-service, ts-route-service, ts-order-service, ts-train-service and ts-auth-service — each had 2 samples in the window exceeding their group-level p95, with earliest such spikes at ~10:26:00 UTC and max values slightly above the p95 baselines.\n"
    "- Disk I/O spikes occurred in ts-route-service, ts-travel-service, ts-auth-service, ts-train-service and ts-order-service; notable high diskio values (e.g., route-service ~196,510 at 10:03:00 UTC).\n"
    "- Other elevated signals: socket, workload, and cpu anomalies (typically 2 samples each) across the services listed above; earliest anomalies for several groups are at or shortly after 10:03:00 UTC.\n\n"
    "Conclusion / next steps: ts-travel-service is the top candidate to investigate first (highest anomaly count and presence of the 'error' metric), followed by ts-route-service and ts-order-service due to large disk I/O and memory anomalies. Investigate memory and disk I/O behavior and related logs/traces on those services around 10:03:00–10:26:00 UTC to identify the root cause."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id  kpi_name  total_count     p95_value  count_in_window  number_of_anomalies earliest_anomaly_timestamp  max_value_in_window
30  ts-travel-service     error           25  0.000000e+00               25                   25  2024-01-24 10:03:00+00:00         0.000000e+00
33  ts-travel-service       mem           25  2.723815e+08               25                    2  2024-01-24 10:26:00+00:00         2.724640e+08
18   ts-route-service       mem           25  2.697714e+08               25                    2  2024-01-24 10:26:00+00:00         2.699056e+08
11   ts-order-service       mem           25  2.680585e+08               25                    2  2024-01-24 10:26:00+00:00         2.680659e+08
25   ts-train-service       mem           25  2.539732e+08               25                    2  2024-01-24 10:26:00+00:00         2.540735e+08
4     ts-auth-service       mem           25  2.458895e+08               25                    2  2024-01-24 10:26:00+00:00         2.459588e+08
15   ts-route-service    diskio           25  8.418870e+04               25                    2  2024-01-24 10:03:00+00:00         1.965107e+05
29  ts-travel-service    diskio           25  6.371736e+04               25                    2  2024-01-24 10:17:00+00:00         6.815016e+04
1     ts-auth-service    diskio           25  5.928099e+04               25                    2  2024-01-24 10:04:00+00:00         6.428521e+04
22   ts-train-service    diskio           25  5.830033e+04               25                    2  2024-01-24 10:22:00+00:00         6.001519e+04
8    ts-order-service    diskio           25  5.823751e+04               25                    2  2024-01-24 10:09:00+00:00         5.979376e+04
20   ts-route-service  workload           25  3.586800e+01               25                    2  2024-01-24 10:04:00+00:00         3.806233e+01
19   ts-route-service    socket           25  2.541667e+01               25                    2  2024-01-24 10:16:00+00:00         2.581667e+01
34  ts-travel-service    socket           25  2.407974e+01               25                    2  2024-01-24 10:24:00+00:00         2.523333e+01
27   ts-train-service  workload           25  2.256504e+01               25                    2  2024-01-24 10:09:00+00:00         2.358320e+01
5     ts-auth-service    socket           25  1.688000e+01               25                    2  2024-01-24 10:14:00+00:00         1.720000e+01
0     ts-auth-service       cpu           25  1.477648e+01               25                    2  2024-01-24 10:07:00+00:00         1.656916e+01
26   ts-train-service    socket           25  1.278333e+01               25                    2  2024-01-24 10:06:00+00:00         1.300000e+01
12   ts-order-service    socket           25  1.231667e+01               25                    2  2024-01-24 10:11:00+00:00         1.290000e+01
35  ts-travel-service  workload           25  1.116139e+01               25                    2  2024-01-24 10:03:00+00:00         1.121146e+01,                   timestamp            cmdb_id  kpi_name          value
0 2024-01-24 10:03:00+00:00   ts-route-service       cpu       4.863806
1 2024-01-24 10:03:00+00:00   ts-route-service    diskio  196510.683525
2 2024-01-24 10:03:00+00:00   ts-train-service       cpu       2.987919
3 2024-01-24 10:03:00+00:00  ts-travel-service     error       0.000000
4 2024-01-24 10:03:00+00:00  ts-travel-service  workload      11.211457
5 2024-01-24 10:04:00+00:00   ts-route-service  workload      35.928883
6 2024-01-24 10:04:00+00:00   ts-route-service    diskio   89798.504605
7 2024-01-24 10:04:00+00:00  ts-travel-service     error       0.000000
8 2024-01-24 10:04:00+00:00   ts-order-service  workload       7.658950
9 2024-01-24 10:04:00+00:00    ts-auth-service    diskio   59659.485650)```
```

In [ ]:
# In[3]:

import pandas as pd

# Ensure df_trace is loaded and timestamps parsed (reuse from kernel if available)
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Target services
services = [
    "ts-auth-service",
    "ts-order-service",
    "ts-route-service",
    "ts-train-service",
    "ts-travel-service"
]

# Build regex to match 'to_' or 'from_' references to any target service
patterns = []
for svc in services:
    # trace_name examples contain substrings like 'to_ts-auth-service' or 'from_ts-auth-service'
    patterns.append(fr"to_{svc}")
    patterns.append(fr"from_{svc}")
regex = "(" + "|".join(patterns) + ")"

# Filter trace rows whose trace_name references the target services
df_tr_sel = df_trace[df_trace['trace_name'].str.contains(regex, regex=True, na=False)].copy()

# Incident window (UTC inclusive)
window_start = pd.Timestamp("2024-01-24 10:00:25", tz="UTC")
window_end = pd.Timestamp("2024-01-24 10:30:25", tz="UTC")

# If no selected traces, return empty frames
if df_tr_sel.empty:
    groups_with_anomalies = pd.DataFrame(columns=[
        'cmdb_id','trace_name','total_count','p95_value','count_in_window',
        'number_of_anomalies','earliest_anomaly_timestamp','max_value_in_window'
    ])
    earliest_anomalies = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value'])
else:
    # 1) Compute global p95 and total_count per (cmdb_id, trace_name) using full series
    grp = df_tr_sel.groupby(['cmdb_id','trace_name'], dropna=False)
    grp_stats = grp['value'].agg(
        total_count='size',
        p95_value=lambda s: float(s.quantile(0.95))
    ).reset_index()

    # 2) Filter rows to the incident window
    mask_window = (df_tr_sel['timestamp'] >= window_start) & (df_tr_sel['timestamp'] <= window_end)
    df_tr_window = df_tr_sel[mask_window].copy()

    # 3) Compute count_in_window per group
    count_in_window = df_tr_window.groupby(['cmdb_id','trace_name']).size().rename('count_in_window').reset_index()

    # Merge group stats with window counts
    groups = grp_stats.merge(count_in_window, on=['cmdb_id','trace_name'], how='left')
    groups['count_in_window'] = groups['count_in_window'].fillna(0).astype(int)

    # Mark anomalies in window: value >= group's p95_value
    df_window_with_p95 = df_tr_window.merge(grp_stats[['cmdb_id','trace_name','p95_value']], on=['cmdb_id','trace_name'], how='left')
    df_window_with_p95['is_anomaly'] = df_window_with_p95['value'] >= df_window_with_p95['p95_value']

    # number_of_anomalies and other summaries per group
    anomalies_count = df_window_with_p95[df_window_with_p95['is_anomaly']].groupby(
        ['cmdb_id','trace_name']
    ).agg(
        number_of_anomalies=('is_anomaly','sum'),
        earliest_anomaly_timestamp=('timestamp','min'),
        max_value_in_window=('value','max')
    ).reset_index()

    # Combine into final groups table
    groups = groups.merge(anomalies_count, on=['cmdb_id','trace_name'], how='left')
    groups['number_of_anomalies'] = groups['number_of_anomalies'].fillna(0).astype(int)
    groups['earliest_anomaly_timestamp'] = groups['earliest_anomaly_timestamp']
    groups['max_value_in_window'] = groups['max_value_in_window'].astype(float)

    # Keep only groups with anomalies > 0 and sort as requested
    groups_with_anomalies = groups[groups['number_of_anomalies'] > 0].copy()
    groups_with_anomalies = groups_with_anomalies.sort_values(
        by=['number_of_anomalies','max_value_in_window'], ascending=[False, False]
    ).head(20)

    # Earliest 20 anomaly rows across all selected groups
    anomalies_all = df_window_with_p95[df_window_with_p95['is_anomaly']].copy()
    earliest_anomalies = anomalies_all[['timestamp','cmdb_id','trace_name','value']].sort_values(
        'timestamp', ascending=True
    ).head(20).reset_index(drop=True)

# Return compact outputs
groups_with_anomalies, earliest_anomalies

```
Out[3]:
```
Summary of the trace analysis (window 2024-01-24 10:00:25–10:30:25 UTC):

- Most flagged "anomalies" are for trace error_rate metrics referencing the target services (to_/from_ traces to ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service). Each of those groups has a global P95 = 0, so every 0.0 value in the window meets the >=P95 criterion — resulting in 25 counts/anomalies per group. This is an artifact of the threshold (p95=0) and not an indication of elevated error rates.
- The earliest anomaly timestamps for these error_rate groups are 2024-01-24 10:03:00 UTC and the recorded values are 0.0 (max_value_in_window = 0.0).
- The only clearly non-zero trace value in the earliest anomaly rows is a root → to_ts-auth-service.duration_mean = 0.351174 s at 10:03:00 UTC. That is a non-zero duration observed at the same early time.
- Interpretation and next steps: trace.error_rate data is not informative here because the baseline p95 is zero (so zeros dominate). Ignore the zero-error "anomalies" and instead focus on non-zero duration/latency traces — in particular investigate traces involving ts-auth-service (duration at ~10:03 UTC) and any duration_p95/duration_mean metrics for ts-travel-service and related services around 10:03 UTC for meaningful latency signals. Also correlate with metric and log signals (CPU, mem, diskio, socket) already observed for deeper diagnosis.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                               trace_name  total_count  p95_value  count_in_window  number_of_anomalies earliest_anomaly_timestamp  max_value_in_window
2                      root      trace.to_ts-auth-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
6                      root     trace.to_ts-order-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
10                     root    trace.to_ts-travel-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
14  ts-admin-travel-service    trace.to_ts-travel-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
18          ts-auth-service    trace.from_ts-auth-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
22          ts-auth-service      trace.to_ts-auth-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
26         ts-basic-service     trace.to_ts-route-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
30         ts-basic-service     trace.to_ts-train-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
34          ts-food-service    trace.to_ts-travel-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
42         ts-order-service   trace.from_ts-order-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
46         ts-order-service  trace.from_ts-travel-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
50         ts-order-service     trace.to_ts-order-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
62         ts-route-service   trace.from_ts-route-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
66         ts-route-service  trace.from_ts-travel-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
70         ts-route-service     trace.to_ts-route-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
74          ts-seat-service  trace.from_ts-travel-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
78          ts-seat-service     trace.to_ts-order-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
82          ts-seat-service    trace.to_ts-travel-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
90       ts-station-service   trace.from_ts-order-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0
94    ts-ticketinfo-service  trace.from_ts-travel-service.error_rate           25        0.0               25                   25  2024-01-24 10:03:00+00:00                  0.0,                    timestamp                cmdb_id                               trace_name     value
0  2024-01-24 10:03:00+00:00                   root   trace.to_ts-auth-service.duration_mean  0.351174
1  2024-01-24 10:03:00+00:00        ts-seat-service     trace.to_ts-order-service.error_rate  0.000000
2  2024-01-24 10:03:00+00:00        ts-seat-service    trace.to_ts-travel-service.error_rate  0.000000
3  2024-01-24 10:03:00+00:00    ts-security-service     trace.to_ts-order-service.error_rate  0.000000
4  2024-01-24 10:03:00+00:00     ts-station-service   trace.from_ts-order-service.error_rate  0.000000
5  2024-01-24 10:03:00+00:00  ts-ticketinfo-service  trace.from_ts-travel-service.error_rate  0.000000
6  2024-01-24 10:03:00+00:00       ts-train-service  trace.from_ts-travel-service.error_rate  0.000000
7  2024-01-24 10:03:00+00:00        ts-seat-service  trace.from_ts-travel-service.error_rate  0.000000
8  2024-01-24 10:03:00+00:00       ts-train-service     trace.to_ts-train-service.error_rate  0.000000
9  2024-01-24 10:03:00+00:00      ts-travel-service     trace.to_ts-order-service.error_rate  0.000000
10 2024-01-24 10:03:00+00:00      ts-travel-service     trace.to_ts-route-service.error_rate  0.000000
11 2024-01-24 10:03:00+00:00      ts-travel-service     trace.to_ts-train-service.error_rate  0.000000
12 2024-01-24 10:03:00+00:00      ts-travel-service    trace.to_ts-travel-service.error_rate  0.000000
13 2024-01-24 10:03:00+00:00     ts-travel2-service     trace.to_ts-route-service.error_rate  0.000000
14 2024-01-24 10:03:00+00:00     ts-travel2-service     trace.to_ts-train-service.error_rate  0.000000
15 2024-01-24 10:03:00+00:00      ts-travel-service  trace.from_ts-travel-service.error_rate  0.000000
16 2024-01-24 10:03:00+00:00       ts-route-service     trace.to_ts-route-service.error_rate  0.000000
17 2024-01-24 10:03:00+00:00       ts-train-service   trace.from_ts-train-service.error_rate  0.000000
18 2024-01-24 10:03:00+00:00       ts-route-service   trace.from_ts-route-service.error_rate  0.000000
19 2024-01-24 10:03:00+00:00       ts-route-service  trace.from_ts-travel-service.error_rate  0.000000)```
```

In [ ]:
# In[4]:

import pandas as pd
import re

# Reuse df_trace if available, otherwise load and parse timestamps
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Incident window (UTC inclusive)
window_start = pd.Timestamp("2024-01-24 10:00:25", tz="UTC")
window_end = pd.Timestamp("2024-01-24 10:30:25", tz="UTC")

# Target services and trace_name keywords
services = [
    "ts-auth-service",
    "ts-order-service",
    "ts-route-service",
    "ts-train-service",
    "ts-travel-service"
]
# compile regex to match 'to_ts-' or 'from_ts-' referencing any target service
svc_pattern = "(" + "|".join(re.escape(s) for s in services) + ")"
direction_pattern = r"(to_|from_).*" + svc_pattern
# keywords for trace types
type_pattern = r"(duration|duration_mean|duration_p95|row_count)"

# Filter by window
mask_window = (df_trace['timestamp'] >= window_start) & (df_trace['timestamp'] <= window_end)

# Filter by service reference in trace_name and by type keywords
mask_service = df_trace['trace_name'].str.contains(direction_pattern, regex=True, na=False)
mask_type = df_trace['trace_name'].str.contains(type_pattern, regex=True, na=False)

df_trace_filtered = df_trace[mask_window & mask_service & mask_type].copy()

# Prepare outputs: (A) up to 20 rows sorted by timestamp ascending with selected columns
trace_rows = df_trace_filtered[['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp', ascending=True).head(20).reset_index(drop=True)

# (B) distinct list of trace_name values present (compact)
trace_names = pd.Series(df_trace_filtered['trace_name'].drop_duplicates().reset_index(drop=True), name='trace_name')

# Return compact results
trace_rows, trace_names

```
Out[4]:
```
summary = (
    "Summary of selected traces in the incident window 2024-01-24 10:00:25–10:30:25 UTC:\n\n"
    "- All returned rows are at 10:03:00 UTC and reference calls to/from the candidate services (auth, order, route, train, travel).\n"
    "- Notable items:\n"
    "  • root → to_ts-auth-service.duration_mean = 0.351 s (the largest non-zero duration seen).\n"
    "  • ts-train-service → from_ts-train-service.row_count = 1512 (high request volume on that path).\n"
    "  • Other duration_p95/duration_mean values are relatively small (mostly milliseconds to a few hundred ms; e.g., station duration_p95 ≈ 0.36 s, ticketinfo duration_p95 ≈ 0.05 s).\n"
    "- Many row_count entries (ts-seat-service, ts-ticketinfo-service, ts-station-service, ts-security-service, etc.) indicate heavy traffic between travel/order/train components at that time.\n"
    "- Distinct trace_name values in the selection: 30 (examples include trace.to_ts-auth-service.duration_mean, trace.from_ts-train-service.row_count, trace.to_ts-order-service.duration_p95).\n\n"
    "Interpretation and next steps: most latencies are low except a noticeable auth-service duration (~0.35 s) and there is high traffic through train/seat paths. Investigate ts-auth-service latency and the high-volume train/seat → travel/order interactions for correlation with the metric anomalies observed earlier."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp                cmdb_id                                  trace_name        value
0  2024-01-24 10:03:00+00:00                   root      trace.to_ts-auth-service.duration_mean     0.351174
1  2024-01-24 10:03:00+00:00       ts-train-service       trace.from_ts-train-service.row_count  1512.000000
2  2024-01-24 10:03:00+00:00       ts-train-service    trace.from_ts-train-service.duration_p95     0.001313
3  2024-01-24 10:03:00+00:00       ts-train-service   trace.from_ts-train-service.duration_mean     0.000917
4  2024-01-24 10:03:00+00:00  ts-ticketinfo-service      trace.from_ts-travel-service.row_count   213.000000
5  2024-01-24 10:03:00+00:00  ts-ticketinfo-service   trace.from_ts-travel-service.duration_p95     0.049198
6  2024-01-24 10:03:00+00:00  ts-ticketinfo-service  trace.from_ts-travel-service.duration_mean     0.017957
7  2024-01-24 10:03:00+00:00     ts-station-service       trace.from_ts-order-service.row_count     5.000000
8  2024-01-24 10:03:00+00:00     ts-station-service    trace.from_ts-order-service.duration_p95     0.359727
9  2024-01-24 10:03:00+00:00     ts-station-service   trace.from_ts-order-service.duration_mean     0.336648
10 2024-01-24 10:03:00+00:00    ts-security-service         trace.to_ts-order-service.row_count     3.000000
11 2024-01-24 10:03:00+00:00       ts-train-service   trace.from_ts-travel-service.duration_p95     0.004461
12 2024-01-24 10:03:00+00:00    ts-security-service      trace.to_ts-order-service.duration_p95     0.008751
13 2024-01-24 10:03:00+00:00        ts-seat-service        trace.to_ts-travel-service.row_count   146.000000
14 2024-01-24 10:03:00+00:00        ts-seat-service     trace.to_ts-travel-service.duration_p95     0.011218
15 2024-01-24 10:03:00+00:00        ts-seat-service    trace.to_ts-travel-service.duration_mean     0.008760
16 2024-01-24 10:03:00+00:00        ts-seat-service         trace.to_ts-order-service.row_count    73.000000
17 2024-01-24 10:03:00+00:00        ts-seat-service      trace.to_ts-order-service.duration_p95     0.011723
18 2024-01-24 10:03:00+00:00        ts-seat-service     trace.to_ts-order-service.duration_mean     0.005976
19 2024-01-24 10:03:00+00:00        ts-seat-service      trace.from_ts-travel-service.row_count    70.000000, 0        trace.to_ts-auth-service.duration_mean
1         trace.to_ts-auth-service.duration_p95
2            trace.to_ts-auth-service.row_count
3       trace.to_ts-order-service.duration_mean
4        trace.to_ts-order-service.duration_p95
                        ...                    
25     trace.from_ts-route-service.duration_p95
26        trace.from_ts-route-service.row_count
27    trace.from_ts-train-service.duration_mean
28     trace.from_ts-train-service.duration_p95
29        trace.from_ts-train-service.row_count
Name: trace_name, Length: 30, dtype: object)```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
# Ensure timestamp parsed (idempotent)
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Candidate services
services = [
    "ts-auth-service",
    "ts-order-service",
    "ts-route-service",
    "ts-train-service",
    "ts-travel-service"
]

# Incident window (UTC inclusive)
window_start = pd.Timestamp("2024-01-24 10:00:25", tz="UTC")
window_end = pd.Timestamp("2024-01-24 10:30:25", tz="UTC")

# Restrict metric series to candidate services (full series used for p95 computation)
df_candidates = df_metric[df_metric['cmdb_id'].isin(services)].copy()

# 1) Compute global p95 per (cmdb_id, kpi_name) using full available series
grp = df_candidates.groupby(['cmdb_id','kpi_name'], dropna=False)
grp_stats = grp['value'].agg(
    total_count='size',
    p95_value=lambda s: float(s.quantile(0.95))
).reset_index()

# 2) Filter rows to the incident window
mask_window = (df_candidates['timestamp'] >= window_start) & (df_candidates['timestamp'] <= window_end)
df_window = df_candidates[mask_window].copy()

# Merge p95_value into window rows for comparison (p95 computed on full series)
df_window = df_window.merge(grp_stats[['cmdb_id','kpi_name','p95_value']], on=['cmdb_id','kpi_name'], how='left')

# 3) Identify anomaly rows in window where value >= group's p95_value
# Keep rows where p95_value is not null (should be present) and value >= p95_value
df_window['is_anomaly'] = df_window['value'] >= df_window['p95_value']

anomalies_all = df_window[df_window['is_anomaly']].copy()

# (A) Earliest 20 anomaly rows across all selected groups sorted by timestamp ascending
anomalies_earliest = anomalies_all[['timestamp','cmdb_id','kpi_name','value','p95_value']].sort_values(
    'timestamp', ascending=True
).head(20).reset_index(drop=True)

# (B) Summary per cmdb_id: total number_of_anomalies, earliest_anomaly_timestamp, max_value_in_window
# Start from list of candidate services to ensure services with zero anomalies are included
service_list_df = pd.DataFrame({'cmdb_id': services})

# Aggregate anomalies by cmdb_id
anomaly_summary_by_service = anomalies_all.groupby('cmdb_id').agg(
    number_of_anomalies=('is_anomaly','sum'),
    earliest_anomaly_timestamp=('timestamp','min'),
    max_value_in_window=('value','max')
).reset_index()

# Merge to include services with zero anomalies
service_summary = service_list_df.merge(anomaly_summary_by_service, on='cmdb_id', how='left')
service_summary['number_of_anomalies'] = service_summary['number_of_anomalies'].fillna(0).astype(int)
service_summary['earliest_anomaly_timestamp'] = service_summary['earliest_anomaly_timestamp']  # may be NaT
service_summary['max_value_in_window'] = service_summary['max_value_in_window']  # may be NaN

# Keep output compact (limit to 20 rows)
anomalies_earliest, service_summary.head(20)

```
Out[5]:
```
summary = (
    "Findings for candidate services in the incident window 2024-01-24 10:00:25–10:30:25 UTC:\n\n"
    "1) Earliest anomaly highlights (first anomalies observed at ~10:03:00 UTC):\n"
    "   - ts-route-service: high disk I/O spike (~196,510) and elevated CPU (4.86 > p95 4.53) at 10:03:00.\n"
    "   - ts-train-service: CPU and socket spikes at 10:03:00 (socket 13 > p95 12.78).\n"
    "   - ts-travel-service: many flagged anomalies including workload and error metrics starting at 10:03:00.\n"
    "   - ts-order-service and ts-auth-service show CPU/diskio/latency anomalies shortly after (10:04–10:06 UTC).\n\n"
    "2) Aggregate anomaly counts and maxima (in-window):\n"
    "   - ts-travel-service: 39 anomalies (earliest 10:03:00), max in-window value ~2.72464e+08 (memory-related metric).\n"
    "   - ts-auth-service: 14 anomalies (earliest 10:04:00), max ~2.45959e+08.\n"
    "   - ts-order-service: 14 anomalies (earliest 10:04:00), max ~2.68066e+08.\n"
    "   - ts-route-service: 14 anomalies (earliest 10:03:00), max ~2.69906e+08.\n"
    "   - ts-train-service: 14 anomalies (earliest 10:03:00), max ~2.54074e+08.\n\n"
    "Notes and interpretation:\n"
    " - ts-travel-service is the top candidate for investigation (highest anomaly count and earliest anomalies at 10:03:00).\n"
    " - There are clear disk I/O and CPU spikes (notably ts-route diskio at 10:03) and memory maxima across services (peaks ~2.4–2.7e8) that warrant investigation.\n"
    " - Some flagged anomalies include metrics with p95=0 (e.g., 'error'), which can produce many trivial matches (0 >= 0). Treat those with caution and prioritize non-zero spikes (cpu, diskio, mem, socket, workload).\n\n"
    "Recommended next steps:\n"
    " - Investigate ts-travel-service first (10:03 start), then ts-route-service and ts-order-service for disk I/O and CPU issues.\n"
    " - Correlate these metric spikes with traces and application logs around 10:03–10:26 UTC to identify root cause (e.g., heavy traffic, a job, or resource pressure)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp            cmdb_id    kpi_name          value     p95_value
0  2024-01-24 10:03:00+00:00   ts-route-service         cpu       4.863806      4.531691
1  2024-01-24 10:03:00+00:00   ts-route-service      diskio  196510.683525  84188.697327
2  2024-01-24 10:03:00+00:00   ts-train-service         cpu       2.987919      2.979506
3  2024-01-24 10:03:00+00:00  ts-travel-service       error       0.000000      0.000000
4  2024-01-24 10:03:00+00:00  ts-travel-service    workload      11.211457     11.161393
5  2024-01-24 10:04:00+00:00   ts-route-service    workload      35.928883     35.867997
6  2024-01-24 10:04:00+00:00   ts-route-service      diskio   89798.504605  84188.697327
7  2024-01-24 10:04:00+00:00  ts-travel-service       error       0.000000      0.000000
8  2024-01-24 10:04:00+00:00   ts-order-service    workload       7.658950      7.330503
9  2024-01-24 10:04:00+00:00    ts-auth-service      diskio   59659.485650  59280.989639
10 2024-01-24 10:04:00+00:00   ts-route-service         cpu       4.610910      4.531691
11 2024-01-24 10:05:00+00:00   ts-order-service         cpu       4.051308      3.665892
12 2024-01-24 10:05:00+00:00   ts-order-service  latency-50       0.009761      0.009587
13 2024-01-24 10:05:00+00:00   ts-order-service  latency-90       0.074448      0.071676
14 2024-01-24 10:05:00+00:00  ts-travel-service       error       0.000000      0.000000
15 2024-01-24 10:06:00+00:00  ts-travel-service       error       0.000000      0.000000
16 2024-01-24 10:06:00+00:00    ts-auth-service  latency-50       0.255701      0.254008
17 2024-01-24 10:06:00+00:00    ts-auth-service  latency-90       0.737470      0.728111
18 2024-01-24 10:06:00+00:00   ts-order-service         cpu       6.573745      3.665892
19 2024-01-24 10:06:00+00:00   ts-train-service      socket      13.000000     12.783333,              cmdb_id  number_of_anomalies earliest_anomaly_timestamp  max_value_in_window
0    ts-auth-service                   14  2024-01-24 10:04:00+00:00         2.459588e+08
1   ts-order-service                   14  2024-01-24 10:04:00+00:00         2.680659e+08
2   ts-route-service                   14  2024-01-24 10:03:00+00:00         2.699056e+08
3   ts-train-service                   14  2024-01-24 10:03:00+00:00         2.540735e+08
4  ts-travel-service                   39  2024-01-24 10:03:00+00:00         2.724640e+08)```
```